# Property Type Prediction - Real Estate Ads Analysis

### Section Overview:
**This module is a core component** of the real estate analytics platform, specifically designed to automatically classify property advertisements into specific categories (e.g., **Apartment Sell**, **Villa Rent**, **Commercial Office**, etc.) based on the textual content of the ads.

### Module Objectives:
- **Text Analysis**: Process Persian real estate advertisement text.
- **Feature Extraction**: Convert text to numerical representations using FastText.
- **Classification**: Predict property category (`cat3_slug`) using multiple machine learning models.
- **Handling Imbalance**: Apply downsampling strategies to manage unequal distribution of property types.

### Technical Implementation:

#### Data Processing Flow:
Raw Text → Persian NLP Processing → Balancing Data → FastText Embeddings → Document Vectors → Classification Models

#### Key Components:
1. **Text Preprocessing**
   - Persian character normalization
   - Tokenization and lemmatization using Hazm
   - Stopword removal and text cleaning

2. **Feature Engineering**
   - FastText word embeddings (300 dimensions)
   - Document vectorization via average word vectors
   - Context Window: 5 words

3. **Classification Models**
   - Logistic Regression
   - Random Forest
   - Linear SVM
   - SGD Classifier

### Module Performance:
- **Best Model**: Linear SVM
- **Accuracy**: ~76.66%
- **Training Data**: Balanced samples (3,000 per category)
- **Feature Dimension**: 300

---
**Note**: This notebook focuses on predicting the **Property Category** (`cat3_slug`), unlike the user-type prediction module.

## Section 1: Environment Setup and Imports

In [3]:
import pandas as pd
import numpy as np
import re
import warnings
from gensim.models import FastText
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from collections import Counter
import fasttext
import os
import string
import time
warnings.filterwarnings("ignore")

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
FIGURES_DIR = PROJECT_ROOT / "figures"

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


**Explanation:**
Import essential libraries for data manipulation (Pandas, Numpy), machine learning (Scikit-Learn), and NLP (Gensim, Hazm). Warnings are suppressed to ensure a clean output during training.

## Section 2: Persian NLP Setup

In [4]:
!pip install git+https://github.com/sobhe/hazm.git
from hazm import *
normalizer = Normalizer()
print(normalizer.normalize("سلاممم  دنیااا"))

  Cloning https://github.com/sobhe/hazm.git to /tmp/pip-req-build-rndrago1
  Running command git clone --filter=blob:none --quiet https://github.com/sobhe/hazm.git /tmp/pip-req-build-rndrago1
  Resolved https://github.com/sobhe/hazm.git to commit d97a5ba13a680a29f2ae5bc6f862e941329890fb
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 43.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 87.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 48.0 MB/s eta 0:00:00
  Created wheel for hazm: filename=hazm-0.10.0-py3-none-any.whl size=893995 sha256=de0c1347cff1fdb85c2ec6d2af0230b173cb09b2eb86fe329a723cf34d31a35d
  Stored in directory: /tmp/pip-ephem-wheel-cache-pbqydfqq/wheels/04/ea/34/e38ffe7a99df974d03e4aa68c3193cd24fa9d341d9f89ffc10
  Created

In [5]:
!pip show hazm

Name: hazm
Version: 0.10.0
Summary: Persian NLP Toolkit
Home-page: https://roshan-ai.ir/hazm/
Author: Roshan
Author-email: salam@roshan-ai.com
License: MIT
Location: /usr/local/lib/python3.11/dist-packages
Requires: fasttext-wheel, flashtext, gensim, nltk, numpy, python-crfsuite, scikit-learn
Required-by: 


**Explanation:**
Install and import `Hazm`, the primary library for processing Persian text. The `Normalizer()` is tested here to ensure character standardization is working correctly.

## Section 3: Data Loading and Exploration

In [ ]:
df = pd.read_csv(DATA_RAW / 'real_estate_ads.csv')

In [ ]:
df.shape

In [ ]:
df_txt = df[['description', 'title', 'cat3_slug', 'user_type']].copy()
print(df_txt["cat3_slug"].value_counts())
print(df["user_type"].value_counts())

In [ ]:
print(df_txt.columns)

df_txt.info()

**Explanation:**
- Load the raw real estate dataset (1 million records).
- Extract relevant text columns (`description`, `title`) and the target label (`cat3_slug`).
- **Insight**: The dataset is highly imbalanced. For example, 'apartment-sell' has ~303k records, while 'workspace' has only ~500. This necessitates data balancing in later steps.

## Section 4: Data Cleaning and Preparation

In [10]:
df_txt.drop_duplicates(inplace=True)
df_txt.dropna(subset=['description', 'title'], inplace=True)

print(df_txt.shape)

(996828, 4)


In [11]:
df_txt['clean_text'] = df_txt['description'] + ' ' + df_txt['title']

**Explanation:**
- **Deduplication**: Remove duplicate rows to prevent data leakage.
- **Null Handling**: Drop rows where text data is missing.
- **Text Fusion**: Combine `description` and `title` into a single feature (`clean_text`) to maximize the information available for the model.

## Section 5: Advanced Text Preprocessing & Cleaning Pipeline

In [12]:
word_counts = Counter(" ".join(df_txt['clean_text']).split())
[word for word, c in word_counts.most_common(200)]

['و',
 'با',
 'در',
 'به',
 'متر',
 'از',
 'واحد',
 'متری',
 'آپارتمان',
 'طبقه',
 'زمین',
 'خواب',
 'تماس',
 'املاک',
 'عالی',
 'دارای',
 'دو',
 'فروش',
 'یک',
 'برای',
 '،',
 'شده',
 'اجاره',
 'شما',
 'خیابان',
 'پارکینگ',
 'تا',
 'های',
 'بزرگ',
 'تک',
 'قیمت',
 '✅',
 'سند',
 ':',
 'منطقه',
 'ملک',
 'امکانات',
 'رهن',
 'بدون',
 'خانه',
 'نقشه',
 'بهترین',
 'فول',
 'جهت',
 'مشاور',
 'بیشتر',
 'خوش',
 'کوچه',
 'دیواری',
 'کابینت',
 'مناسب',
 'کامل',
 'دسترسی',
 'انباری',
 'قابل',
 'اطلاعات',
 'نور',
 '.',
 'بازدید',
 'لوکیشن',
 'ساخت',
 'هم',
 '۲',
 'خرید',
 'بگیرید',
 'سلام',
 'مغازه',
 '/',
 'بسیار',
 'ها',
 'ویلایی',
 'کف',
 'سه',
 'گاز',
 'آب',
 'شهرک',
 'واقع',
 'سرامیک',
 'میباشد',
 'رو',
 'واحدی',
 'برگ',
 'اول',
 'ما',
 'حیاط',
 'ویلا',
 'فقط',
 'کمد',
 'برق',
 'آدرس',
 'ای',
 'باغ',
 'مسکونی',
 '۳',
 'بر',
 'که',
 'سالن',
 'بلوار',
 'اتاق',
 'دارد',
 'موقعیت',
 'تمیز',
 'اصلی',
 'ساختمان',
 'آسانسور',
 'شیک',
 '⚜️',
 'مشابه',
 'سرمایه',
 'هر',
 'می',
 'معاوضه',
 'متراژ',
 'خو

In [13]:
def preprocess_text(text):
    if text is None:
        return ""
    text = str(text)
    # Persian-Arabic character standardization
    replacements = {
        'ك': 'ک', 'دِ': 'د', 'بِ': 'ب', 'زِ': 'ز', 'ذِ': 'ذ', 'شِ': 'ش', 'سِ': 'س', 'ى': 'ی',
        'ي': 'ی', '١': '1', '٢': '2', '٣': '3', '٤': '4', '٥': '5', '٦': '6', '٧': '7', '٨': '8', '٩': '9', '٠': '0',
        '۱': '1', '۲': '2', '۳': '3', '۴': '4', '۵': '5', '۶': '6', '۷': '7', '۸': '8', '۹': '9', '۰': '0'
    }

    # Apply character replacements
    for k, v in replacements.items():
        text = text.replace(k, v)

    # Remove zero-width and control characters
    zero_width_chars = [
        "\u200c", "\u200b", "\u200d", "\uFEFF", "\u2060",
        "\u202F", "\u200f", "\u202a", "\u202e"
    ]
    for zw in zero_width_chars:
        text = text.replace(zw, " ")

    # Clean social media symbols and special characters
    text = re.sub(r"[#@]", " ", text)
    text = re.sub(r"[^a-zA-Z0-9\u0600-\u06FF\s]", " ", text)

    # Reduce letter repetitions
    text = re.sub(r"(.)\1{2,}", r"\1", text)

    # Normalize units
    text = re.sub(r"(\d+)\s+(متر|اتاق|سال|طبقه|خواب)", r"\1\2", text)

    # Clean extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

# --- Function test ---
sample = "فروش فوری #ویلا ی 120   متر در خیابان  آزادی!!! @admin تلفن: ۰۹۱۲۳۴۵۶۷۸۹ خیلیییی تمیز"
print(preprocess_text(sample))

فروش فوری ویلا ی 120متر در خیابان آزادی admin تلفن 09123456789 خیلی تمیز


In [14]:
# Load Persian stopwords
stopwords = pd.read_csv(DATA_RAW / 'Stopwords.csv', header=0)
stops = set(stopwords['word'])
stops.update([
    'تماس', 'شما', 'باسلام', 'سلام', 'درود', 'هماهنگی', 'مشاهده', 'آگهی',
    'کلیک', 'اطلاعات', 'لطفا', 'جهت', 'بازدید', 'مورد', 'بابت'
])
# normalizer = Normalizer()
tokenizer = WordTokenizer()
lemmatizer = Lemmatizer()


def clean_text(raw_text):
    text = preprocess_text(raw_text)
    words = tokenizer.tokenize(text)

    meaningful_words = []
    for w in words:
        lemma = lemmatizer.lemmatize(w)

        # Handle Hazm lemmatization artifacts
        if '#' in lemma:
            lemma = lemma.replace('#', ' ')

        # Filter stopwords
        if (lemma not in stops) and (w not in stops):
            meaningful_words.append(lemma)
    return " ".join(meaningful_words)

In [15]:
df_txt['clean_text'] = df_txt['clean_text'].apply(clean_text)

**Explanation:**
This pipeline transforms raw text into clean tokens:
1.  **Stopword Identification**: Calculate word frequencies to find common, low-value words.
2.  **Normalization**: Standardize Persian characters and remove zero-width spaces.
3.  **Lemmatization**: Convert words to their root form using `Hazm`.
4.  **Filtering**: Remove stopwords, numbers, and special characters to reduce noise.

In [16]:
# Finding Stop Words
word_counts = Counter(" ".join(df_txt['clean_text']).split())
[word for word, c in word_counts.most_common(200)]

['،',
 'متر',
 'متری',
 'خواب',
 'طبقه',
 'واحد',
 'املاک',
 'آپارتمان',
 'زمین',
 'دارای',
 '2',
 'پارکینگ',
 'فروش',
 'عالی',
 'ویلا',
 'خیابان',
 'اجاره',
 '3',
 'سند',
 'تک',
 'بزرگ',
 'قیمت',
 'منطقه',
 'ملک',
 'امکانات',
 'فول',
 'رهن',
 'مشاور',
 'کابینت',
 'انباری',
 'دسترسی',
 'بهترین',
 'نقشه',
 'خانه',
 'مناسب',
 'لوکیشن',
 'کوچه',
 'کف',
 'خرید',
 'دیواری',
 'شهرک',
 'نور',
 'کامل',
 'گاز',
 'ساخت',
 'میباشد',
 'آدرس',
 '1',
 '4',
 'سالن',
 'واحدی',
 'سه',
 'آب',
 '5',
 'میلیون',
 'آسانسور',
 'مغازه',
 'سرامیک',
 'کمد',
 'برگ',
 'حیاط',
 'واقع',
 'برق',
 'متراژ',
 'فایل',
 'ساختمان',
 'اول',
 'اتاق',
 'موقعیت',
 'خوابه',
 'بلوار',
 'باغ',
 'مسکونی',
 'فاز',
 'گذار',
 'سرمایه',
 'نورگیر',
 'معاوضه',
 'تمیز',
 'دفتر',
 'اصلی',
 '10',
 'بالکن',
 'سرویس',
 'اپارتمان',
 'بازسازی',
 'نما',
 'مشابه',
 '6',
 'گذارد',
 '100',
 'تخلیه',
 'قابلیت',
 'شهر',
 'جنوبی',
 'موجود',
 'تراس',
 'مستر',
 'اختصاصی',
 'فوق',
 'درب',
 'نوساز',
 'شمالی',
 'تجاری',
 'مسکن',
 'وام',
 'کار',
 'کارشناس

## Section 6: Data Balancing and FastText Preparation

In [17]:
# Remove records with missing user_type
df_txt_cleaned = df_txt.dropna(subset=['cat3_slug']).copy()

# Balanced sampling - 30K from each class
N = 3000

df_small = (
    df_txt_cleaned
    .groupby("cat3_slug")
    .apply(lambda x: x.sample(min(len(x), N), random_state=42))
    .reset_index(drop=True)
)

print(df_small["cat3_slug"].value_counts())
print("تعداد کل نمونه‌ها:", len(df_small))

# Prepare sentences for FastText training
sentences = [text.split() for text in df_small['clean_text']]
print(f"تعداد کل نمونه‌ها: {len(sentences)}")
print(f"تعداد کلمات در اولین سند: {len(sentences[0])}")

# Format data for supervised FastText
# df_txt_cleaned["ft_format"] = '__label__' + df_txt_cleaned["cat3_slug"].astype(str) + ' ' + df_txt_cleaned["clean_text"].astype(str)
df_small["ft_format"] = '__label__' + df_small["cat3_slug"].astype(str) + ' ' + df_small["clean_text"].astype(str)

# Train-test split with stratification
# train_df, test_df = train_test_split(df_txt_cleaned, test_size=0.2, random_state=42, stratify=df_txt_cleaned["cat3_slug"])
train_df, test_df = train_test_split(df_small, test_size=0.2, random_state=42, stratify=df_small["cat3_slug"])

train_file = 'fasttext_train.txt'
test_file = 'fasttext_test.txt'

train_df['ft_format'].to_csv(train_file, index=False, header=False, encoding='utf-8')
test_df['ft_format'].to_csv(test_file, index=False, header=False, encoding='utf-8')

print(f"فایل آموزش در مسیر {train_file} ذخیره شد. تعداد نمونه‌ها: {len(train_df)}")


cat3_slug
apartment-rent                        3000
apartment-sell                        3000
house-villa-rent                      3000
house-villa-sell                      3000
industry-agriculture-business-rent    3000
industry-agriculture-business-sell    3000
office-rent                           3000
office-sell                           3000
partnership                           3000
plot-old                              3000
presell                               3000
shop-rent                             3000
shop-sell                             3000
suite-apartment                       3000
villa                                 3000
workspace                              537
Name: count, dtype: int64
تعداد کل نمونه‌ها: 45537
تعداد کل نمونه‌ها: 45537
تعداد کلمات در اولین سند: 62
فایل آموزش در مسیر fasttext_train.txt ذخیره شد. تعداد نمونه‌ها: 36429


**Explanation:**
- **Balancing**: Due to severe class imbalance, we downsample the data, selecting **3,000 samples** per category. This prevents the model from being biased toward the majority classes (like apartments).
- **FastText Formatting**: Convert data into the specific format required by FastText: `__label__category text`.
- **Splitting**: Segregate data into Training (80%) and Testing (20%) sets using stratified sampling.

## Section 7: Word Embedding Training

In [18]:
# import multiprocessing
# WORKERS = multiprocessing.cpu_count()
VECTOR_SIZE = 300
MIN_COUNT = 5
WORKERS = 8
EPOCHS = 20

print(f"\nدر حال آموزش مدل FastText با ابعاد {VECTOR_SIZE}...")
start_time = time.time()

fasttext_model = FastText(
    sentences,
    vector_size=VECTOR_SIZE,
    window=5,
    min_count=MIN_COUNT,
    workers=WORKERS,
    sg=1,
    epochs=EPOCHS 
)

training_time = time.time() - start_time
print(f"آموزش FastText در زمان {training_time:.2f} ثانیه به پایان رسید.")
print(f"اندازه واژگان مدل: {len(fasttext_model.wv)}")


در حال آموزش مدل FastText با ابعاد 300...
آموزش FastText در زمان 288.34 ثانیه به پایان رسید.
اندازه واژگان مدل: 11158


**Explanation:**
Train a specific **FastText** model on the real estate corpus.
- **Vector Size**: 300 dimensions (captures deep semantic relationships).
- **Window**: 5 words (context size).
- **Result**: The model learns vector representations where semantically similar real estate terms are mathematically close.

## Section 8: Document Vectorization

In [19]:
def document_vector(model, doc_words):
    # Filter words present in vocabulary
    words = [word for word in doc_words if word in model.wv]
    
    if len(words) >= 1:
        # Average of word vectors
        return np.mean(model.wv[words], axis=0)
    else:
        # Zero vector for empty documents
        return np.zeros(model.vector_size)
print("تبدیل اسناد به بردار میانگین...")
X = np.array([document_vector(fasttext_model, doc) for doc in sentences])
# y = df_txt_cleaned["cat3_slug"]
y = df_small["cat3_slug"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y
)

print(f"ابعاد داده‌های آموزشی: {X_train.shape}")
print(f"ابعاد داده‌های تست: {X_test.shape}")

تبدیل اسناد به بردار میانگین...
ابعاد داده‌های آموزشی: (36429, 300)
ابعاد داده‌های تست: (9108, 300)


**Explanation:**
- **Document Vectors**: Convert every advertisement into a single numerical vector by averaging the vectors of its constituent words.
- **Output**:
    - Training Set: ~36,000 vectors (300 dimensions).
    - Test Set: ~9,000 vectors (300 dimensions).

## Section 9: Model Training and Evaluation

In [20]:
# Encode string labels to numerical values
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

# Define multiple classification models

models = {
    'Logistic Regression': LogisticRegression(
        random_state=42,
        class_weight='balanced',
        max_iter=1000
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight='balanced',
        n_jobs=WORKERS
    ),
    'Linear SVM': SVC(
        kernel='linear',
        random_state=42,
        class_weight='balanced',
    ),
    'SGD Classifier (Optimized)': SGDClassifier(
    loss='hinge',
    alpha=0.0001,
    max_iter=1000, 
    n_jobs=-1,
    random_state=42,
    class_weight='balanced'
)
}

# Train and evaluate all models

results = {}

for name, model in models.items():
    print(f"\n--- آموزش مدل {name} ---")
    start_time_model = time.time()
    
    # Model training
    model.fit(X_train, y_train_enc)
    
    # Predicting
    y_pred = model.predict(X_test)
    
    accuracy = accuracy_score(y_test_enc, y_pred)
    report = classification_report(y_test_enc, y_pred)
    cm = confusion_matrix(y_test_enc, y_pred)
    
    results[name] = {
        'accuracy': accuracy,
        'report': report,
        'confusion_matrix': cm
    }
    
    print(f"دقت مدل {name}: {accuracy:.4f}")
    print(f"زمان اجرا: {time.time() - start_time_model:.2f} ثانیه")
    print(report)


--- آموزش مدل Logistic Regression ---
دقت مدل Logistic Regression: 0.7513
زمان اجرا: 26.48 ثانیه
              precision    recall  f1-score   support

           0       0.66      0.70      0.68       600
           1       0.67      0.71      0.69       600
           2       0.72      0.73      0.73       600
           3       0.76      0.72      0.74       600
           4       0.77      0.81      0.79       600
           5       0.69      0.68      0.68       600
           6       0.74      0.65      0.69       600
           7       0.76      0.70      0.73       600
           8       0.84      0.81      0.82       600
           9       0.73      0.73      0.73       600
          10       0.82      0.82      0.82       600
          11       0.73      0.74      0.73       600
          12       0.77      0.76      0.76       600
          13       0.87      0.87      0.87       600
          14       0.89      0.86      0.87       600
          15       0.39      0.69    

**Explanation:**
Train and evaluate four distinct classifiers:
1.  **Logistic Regression**
2.  **Random Forest**
3.  **Linear SVM**
4.  **SGD Classifier (Optimized)**

All models use `class_weight='balanced'` to handle any remaining minor imbalances. The results (Precision, Recall, F1-Score) show that linear models (SVM/SGD) generally outperform tree-based models for this high-dimensional text data.

## Section 10: Results Analysis

In [22]:
best_model_name = max(results.keys(), key=lambda x: results[x]['accuracy'])
print(f"بهترین مدل: {best_model_name} با دقت {results[best_model_name]['accuracy']:.4f}")

بهترین مدل: Linear SVM با دقت 0.7672


**Explanation:**
- **Model Selection**: Automatically identifies the model with the highest accuracy score.
- **Winner**: The **Linear SVM** achieved the best performance (Accuracy: ~76.66%), making it the optimal choice for deployment in the production environment.